In [2]:
from google.colab import files
uploaded = files.upload()


Saving crop_yield.csv to crop_yield.csv


In [3]:
import pandas as pd

file_name = next(iter(uploaded))
df_raw = pd.read_csv(file_name, skipinitialspace=False)
df = df_raw.copy()

print("Rows and columns:", df.shape)
print("Blank cells:", int(df.isna().sum().sum()))
print("Exact duplicate rows:", int(df.duplicated().sum()))

for column in ["Crop", "Season"]:
    text = df[column].astype(str)
    has_extra_spaces = text != text.str.strip()

    print(f"\n{column}")
    print("Rows with extra spaces:", int(has_extra_spaces.sum()))
    print("Examples:", text[has_extra_spaces].drop_duplicates().head(5).tolist())

print("\nRecords by year:")
print(df.groupby("Crop_Year").size().to_string())

Rows and columns: (19689, 10)
Blank cells: 0
Exact duplicate rows: 0

Crop
Rows with extra spaces: 172
Examples: ['Coconut ']

Season
Rows with extra spaces: 19689
Examples: ['Whole Year ', 'Kharif     ', 'Rabi       ', 'Autumn     ', 'Summer     ']

Records by year:
Crop_Year
1997     410
1998     646
1999     650
2000     772
2001     750
2002     815
2003     801
2004     787
2005     811
2006     823
2007     828
2008     847
2009     865
2010     859
2011     907
2012     894
2013     976
2014     965
2015     998
2016    1036
2017    1070
2018    1063
2019    1079
2020      37


In [4]:
df_clean = df_raw.copy()

for column in ["Crop", "Season"]:
    before_unique = df_clean[column].nunique()

    df_clean[column] = df_clean[column].astype("string").str.strip()

    after_unique = df_clean[column].nunique()
    remaining_spaces = int(
        (df_clean[column] != df_clean[column].str.strip()).sum()
    )

    print(f"\n{column}")
    print("Unique labels before:", before_unique)
    print("Unique labels after:", after_unique)
    print("Rows still containing extra spaces:", remaining_spaces)

print("\nRows and columns after cleaning:", df_clean.shape)
print("Exact duplicates after label cleaning:", int(df_clean.duplicated().sum()))


Crop
Unique labels before: 55
Unique labels after: 55
Rows still containing extra spaces: 0

Season
Unique labels before: 6
Unique labels after: 6
Rows still containing extra spaces: 0

Rows and columns after cleaning: (19689, 10)
Exact duplicates after label cleaning: 0


In [5]:
numeric_columns = [
    "Area",
    "Production",
    "Annual_Rainfall",
    "Fertilizer",
    "Pesticide",
    "Yield"
]

range_audit = pd.DataFrame({
    "data_type": df_clean[numeric_columns].dtypes.astype(str),
    "minimum": df_clean[numeric_columns].min(),
    "maximum": df_clean[numeric_columns].max(),
    "zero_count": (df_clean[numeric_columns] == 0).sum(),
    "negative_count": (df_clean[numeric_columns] < 0).sum()
})

print(range_audit)

                data_type  minimum       maximum  zero_count  negative_count
Area              float64     0.50  5.080810e+07           0               0
Production          int64     0.00  6.326000e+09         112               0
Annual_Rainfall   float64   301.30  6.552700e+03           0               0
Fertilizer        float64    54.17  4.835407e+09           0               0
Pesticide         float64     0.09  1.575051e+07           0               0
Yield             float64     0.00  2.110500e+04         112               0


In [6]:
zero_production = df_clean["Production"].eq(0)
zero_yield = df_clean["Yield"].eq(0)

print("Production is zero:", int(zero_production.sum()))
print("Yield is zero:", int(zero_yield.sum()))
print("Both are zero:", int((zero_production & zero_yield).sum()))
print("Production zero, yield non-zero:", int((zero_production & ~zero_yield).sum()))
print("Yield zero, production non-zero:", int((zero_yield & ~zero_production).sum()))

print("\nExample rows where either value is zero:")
print(
    df_clean.loc[
        zero_production | zero_yield,
        ["Crop", "Crop_Year", "Season", "Area", "Production", "Yield"]
    ].head(10).to_string(index=False)
)


Production is zero: 112
Yield is zero: 112
Both are zero: 108
Production zero, yield non-zero: 4
Yield zero, production non-zero: 4

Example rows where either value is zero:
          Crop  Crop_Year     Season     Area  Production  Yield
      Cardamom       1997 Whole Year   2674.0           7    0.0
     Sunflower       1997 Whole Year    278.0           1    0.0
      Soyabean       2002     Kharif  41284.0           0    0.0
other oilseeds       2002 Whole Year   2253.0           0    0.0
      Arecanut       2002 Whole Year   4956.0           0    0.0
other oilseeds       2003       Rabi  14609.0           0    0.0
      Arecanut       2003 Whole Year   5219.0           0    0.0
    Niger seed       2004     Kharif   1239.0           0    0.0
        Garlic       2005 Whole Year    800.0           0    0.0
         Maize       2005     Kharif 202830.0           0    0.0


In [7]:
mismatch = zero_production ^ zero_yield

mismatch_rows = df_clean.loc[
    mismatch,
    ["Crop", "Crop_Year", "Season", "Area", "Production", "Yield"]
].copy()

mismatch_rows["Production_per_Area"] = (
    mismatch_rows["Production"] / mismatch_rows["Area"]
)

print(mismatch_rows.to_string(index=False))

        Crop  Crop_Year     Season     Area  Production  Yield  Production_per_Area
    Cardamom       1997 Whole Year   2674.0           7    0.0             0.002618
   Sunflower       1997 Whole Year    278.0           1    0.0             0.003597
Cotton(lint)       2017     Kharif 138290.0         337    0.0             0.002437
Cotton(lint)       2017       Rabi  43341.0         101    0.0             0.002330
     Tobacco       2012     Kharif      1.0           0    0.4             0.000000
     Tobacco       2014     Kharif      1.0           0    0.4             0.000000
  Horse-gram       2016       Rabi      1.0           0    0.4             0.000000
      Masoor       2004     Kharif      0.5           0    0.4             0.000000


In [8]:
numeric_columns = [
    "Area",
    "Production",
    "Annual_Rainfall",
    "Fertilizer",
    "Pesticide",
    "Yield"
]

outlier_results = []

for column in numeric_columns:
    q1 = df_clean[column].quantile(0.25)
    q3 = df_clean[column].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    flagged = (
        (df_clean[column] < lower_bound) |
        (df_clean[column] > upper_bound)
    )

    outlier_results.append({
        "column": column,
        "Q1": q1,
        "Q3": q3,
        "lower_bound": lower_bound,
        "upper_bound": upper_bound,
        "flagged_rows": int(flagged.sum())
    })

print(pd.DataFrame(outlier_results).to_string(index=False))

         column        Q1           Q3   lower_bound  upper_bound  flagged_rows
           Area   1390.00 7.511200e+04 -1.091930e+05 1.856950e+05          3076
     Production   1393.00 1.227180e+05 -1.805945e+05 3.047055e+05          3373
Annual_Rainfall    940.70 1.643700e+03 -1.138000e+02 2.698200e+03          1527
     Fertilizer 188014.62 1.000385e+07 -1.453573e+07 2.472760e+07          3093
      Pesticide    356.70 2.004170e+04 -2.917080e+04 4.956920e+04          3036
          Yield      0.60 2.388889e+00 -2.083333e+00 5.072222e+00          3065


In [9]:
for column in numeric_columns:
    q1 = df_clean[column].quantile(0.25)
    q3 = df_clean[column].quantile(0.75)
    upper_bound = q3 + 1.5 * (q3 - q1)

    examples = df_clean[df_clean[column] > upper_bound].nlargest(3, column)

    print(f"\nLargest flagged values for {column}:")
    print(
        examples[["Crop", "Crop_Year", "Season", column]]
        .to_string(index=False)
    )


Largest flagged values for Area:
      Crop  Crop_Year Season       Area
Niger seed       1997   Rabi 50808100.0
     Wheat       2019   Rabi 10216517.0
     Wheat       2016   Rabi  9884913.0

Largest flagged values for Production:
   Crop  Crop_Year     Season  Production
Coconut       2005 Whole Year  6326000000
Coconut       2011 Whole Year  6200900000
Coconut       2006 Whole Year  6054000000

Largest flagged values for Annual_Rainfall:
     Crop  Crop_Year     Season  Annual_Rainfall
 Arecanut       1998 Whole Year           6552.7
Arhar/Tur       1998     Kharif           6552.7
   Banana       1998 Whole Year           6552.7

Largest flagged values for Fertilizer:
      Crop  Crop_Year Season   Fertilizer
Niger seed       1997   Rabi 4835406877.0
     Wheat       2019   Rabi 1754788960.0
     Wheat       2019   Rabi 1692266087.0

Largest flagged values for Pesticide:
      Crop  Crop_Year Season   Pesticide
Niger seed       1997   Rabi 15750511.00
     Wheat       2019   Rabi

In [11]:
columns_to_review = [
    "Crop", "Crop_Year", "Season",
    "Area", "Production", "Annual_Rainfall",
    "Fertilizer", "Pesticide", "Yield"
]

niger_example = df_clean[
    (df_clean["Crop"] == "Niger seed") &
    (df_clean["Crop_Year"] == 1997) &
    (df_clean["Season"] == "Rabi")
]

coconut_examples = df_clean[
    (df_clean["Crop"] == "Coconut") &
    (df_clean["Crop_Year"].isin([2005, 2014, 2019]))
]

print("Niger seed example:")
print(niger_example[columns_to_review].to_string(index=False))

print("\nCoconut examples:")
print(coconut_examples[columns_to_review].to_string(index=False))

Niger seed example:
      Crop  Crop_Year Season       Area  Production  Annual_Rainfall   Fertilizer  Pesticide    Yield
Niger seed       1997   Rabi 50808100.0    38657300           1852.9 4835406877.0 15750511.0 0.698889

Coconut examples:
   Crop  Crop_Year     Season     Area  Production  Annual_Rainfall   Fertilizer  Pesticide        Yield
Coconut       2005 Whole Year 104438.0   891991928        1075.5000  12524204.96   21931.98  7717.767778
Coconut       2005 Whole Year  19057.0   204937000        2184.6000   2285315.44    4001.97  8486.539565
Coconut       2005 Whole Year 395885.0  1606890000        1316.2000  47474529.20   83135.85  4165.660741
Coconut       2005 Whole Year 897833.0  6326000000        3030.9000 107668133.40  188544.93  6500.859286
Coconut       2005 Whole Year 370515.0  4867100000        1364.1000  44432158.80   77808.15 11777.931720
Coconut       2005 Whole Year  24884.0   323525800        1650.1000   2984089.28    5225.64 11972.035000
Coconut       2014 Who

In [12]:
comparison = df_clean[
    (df_clean["Area"] > 0) &
    (df_clean["Production"] > 0)
].copy()

comparison["calculated_production_per_area"] = (
    comparison["Production"] / comparison["Area"]
)

comparison["relative_difference"] = (
    (comparison["Yield"] - comparison["calculated_production_per_area"]).abs()
    / comparison["calculated_production_per_area"]
)

print("Records compared:", len(comparison))
print(
    "Records within 1% of Production / Area:",
    int((comparison["relative_difference"] <= 0.01).sum())
)
print(
    "Median relative difference:",
    round(comparison["relative_difference"].median(), 4)
)

print("\nExamples with the largest relative differences:")
print(
    comparison.nlargest(5, "relative_difference")[
        ["Crop", "Crop_Year", "Season", "Area", "Production",
         "Yield", "calculated_production_per_area", "relative_difference"]
    ].to_string(index=False)
)


Records compared: 19577
Records within 1% of Production / Area: 4050
Median relative difference: 0.0505

Examples with the largest relative differences:
      Crop  Crop_Year     Season    Area  Production      Yield  calculated_production_per_area  relative_difference
 Cashewnut       2008 Whole Year 99168.0       51667 338.548966                        0.521005           648.800140
 Arhar/Tur       2009     Winter   441.0         764  68.230000                        1.732426            38.384071
Niger seed       2013     Kharif 63611.0       11567   4.987647                        0.181840            26.428825
 Sunflower       2016     Kharif  2273.0           7   0.054706                        0.003080            16.763781
 Sunflower       2006     Kharif  6493.0        2724   5.020000                        0.419529            10.965808


In [13]:
for tolerance in [0.01, 0.05, 0.10]:
    matches = comparison["relative_difference"] <= tolerance
    print(
        f"Within {tolerance:.0%}: "
        f"{int(matches.sum())} of {len(comparison)} "
        f"({matches.mean():.1%})"
    )

Within 1%: 4050 of 19577 (20.7%)
Within 5%: 9732 of 19577 (49.7%)
Within 10%: 13402 of 19577 (68.5%)


In [14]:
df_prepared = df_raw.copy()

for column in ["Crop", "Season"]:
    df_prepared[column] = (
        df_prepared[column].astype("string").str.strip()
    )

print("Original shape:", df_raw.shape)
print("Prepared shape:", df_prepared.shape)
print("Blank cells:", int(df_prepared.isna().sum().sum()))
print("Exact duplicate rows:", int(df_prepared.duplicated().sum()))
print(
    "Crop labels with extra spaces:",
    int((df_prepared["Crop"] != df_prepared["Crop"].str.strip()).sum())
)
print(
    "Season labels with extra spaces:",
    int((df_prepared["Season"] != df_prepared["Season"].str.strip()).sum())
)

output_path = "/content/crop_yield_prepared.csv"
df_prepared.to_csv(output_path, index=False)
print("Saved prepared file to:", output_path)

Original shape: (19689, 10)
Prepared shape: (19689, 10)
Blank cells: 0
Exact duplicate rows: 0
Crop labels with extra spaces: 0
Season labels with extra spaces: 0
Saved prepared file to: /content/crop_yield_prepared.csv


In [15]:
crop_yield_outliers = []

for crop, group in df_prepared.groupby("Crop"):
    values = group["Yield"]
    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    flagged = (values < lower) | (values > upper)

    crop_yield_outliers.append({
        "Crop": crop,
        "records": len(group),
        "yield_lower_fence": lower,
        "yield_upper_fence": upper,
        "flagged_yield_records": int(flagged.sum()),
        "flagged_percent": round(100 * flagged.mean(), 1)
    })

crop_yield_outliers = pd.DataFrame(crop_yield_outliers)
print(
    crop_yield_outliers
    .sort_values("flagged_percent", ascending=False)
    .head(10)
    .to_string(index=False)
)

               Crop  records  yield_lower_fence  yield_upper_fence  flagged_yield_records  flagged_percent
           Sannhamp      160          -0.734167           1.872500                     35             21.9
           Cardamom       74          -0.010223           0.181563                     13             17.6
     Oilseeds total       29          -0.072885           2.774808                      5             17.2
           Arecanut      162          -0.266809           2.950085                     21             13.0
       Black pepper      126          -0.914196           2.116518                     16             12.7
          Guar seed       63           0.423423           1.113342                      8             12.7
     other oilseeds      126          -0.245179           1.739107                     14             11.1
          Cashewnut      134          -0.226888           1.275406                     14             10.4
Other Summer Pulses       10         

In [16]:
for crop in ["Sannhamp", "Cardamom", "Arecanut"]:
    group = df_prepared[df_prepared["Crop"] == crop]
    q1 = group["Yield"].quantile(0.25)
    q3 = group["Yield"].quantile(0.75)
    upper_fence = q3 + 1.5 * (q3 - q1)

    examples = group[group["Yield"] > upper_fence].nlargest(3, "Yield")

    print(f"\n{crop} — largest values above its upper fence ({upper_fence:.4f}):")
    print(
        examples[["Crop", "Crop_Year", "Season", "Area", "Production", "Yield"]]
        .to_string(index=False)
    )


Sannhamp — largest values above its upper fence (1.8725):
    Crop  Crop_Year     Season   Area  Production    Yield
Sannhamp       2008     Kharif 3358.0       14978 6.622609
Sannhamp       2017     Kharif  745.0        3638 5.828000
Sannhamp       2006 Whole Year 1294.0        7252 5.581250

Cardamom — largest values above its upper fence (0.1816):
    Crop  Crop_Year     Season   Area  Production    Yield
Cardamom       2008 Whole Year  245.0         704 2.870000
Cardamom       2006     Kharif  900.0        1440 1.612000
Cardamom       2015 Whole Year 3983.0         307 0.653636

Arecanut — largest values above its upper fence (2.9501):
    Crop  Crop_Year     Season     Area  Production     Yield
Arecanut       2010 Whole Year 211265.0     2284410 10.948182
Arecanut       2013 Whole Year 218007.0     2428335 10.930000
Arecanut       2017 Whole Year 279459.0     3030925 10.920000
